# 食生活が似ている都市を探す

## 1. データセットの確認

**SSDSE-C-2026：47都市 × 食料212品目。** 二人以上の世帯の1世帯当たり年間支出額（円）、
2023～2025年平均。東京都は東京都区部。

| カテゴリ | 品目数 | 品目例 |
|---|---:|---|
| 穀類 | 13 | 米、食パン |
| 魚介類 | 35 | まぐろ、さけ |
| 肉類 | 9 | 牛肉、豚肉 |
| 乳卵類 | 7 | 牛乳、卵 |
| 野菜・海藻 | 47 | キャベツ、納豆 |
| 果物 | 15 | りんご、みかん |
| 油脂・調味料 | 18 | 食用油、しょう油 |
| 菓子類 | 16 | せんべい、チョコレート |
| 調理食品 | 17 | 弁当、ぎょうざ |
| 飲料 | 14 | 緑茶、コーヒー |
| 酒類 | 8 | 清酒、ビール |
| 外食 | 13 | 和食、喫茶代、学校給食 |

出典：[統計センター SSDSE-C-2026](https://www.nstac.go.jp/use/literacy/ssdse/)（元統計：家計調査）。[公式解説](https://www.nstac.go.jp/files/kaisetsu-C-2026.pdf)。


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy.cluster.hierarchy import linkage, dendrogram, cut_tree, leaves_list
from sklearn.preprocessing import StandardScaler
from IPython.display import display

# 日本語の表示設定
if "Noto Sans CJK JP" in {f.name for f in font_manager.fontManager.ttflist}:
    plt.rcParams["font.family"] = "Noto Sans CJK JP"
else:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "setuptools", "japanize-matplotlib==1.1.3",
    ])
    import japanize_matplotlib
    japanize_matplotlib.japanize()

plt.rcParams["axes.unicode_minus"] = False

# 公式CSV（同名のローカルファイルがあればそれを使用）
path = Path("SSDSE-C-2026.csv")
source = path if path.exists() else "https://www.nstac.go.jp/files/SSDSE-C-2026.csv"
raw = pd.read_csv(source, encoding="cp932")

# 項目名の行と「全国」を除き、都市名を行名にする
item_names = raw.iloc[0]
data = raw.iloc[1:].set_index("City").drop(index="全国")

# 個別品目だけを選ぶ（合計・小計・世帯人員は除く）
amounts = data.filter(regex=r"^LB[0-9]{6}$").apply(pd.to_numeric)
amounts = amounts.rename(columns=item_names)
amounts.index.name = "都市"

assert amounts.shape == (47, 212), "2026年版の47都市・212品目を確認してください。"
print(f"{amounts.shape[0]}都市 × {amounts.shape[1]}品目（年間支出額：円）")

# 5都市・12品目をランダム表示。分類には全都市・全品目を使う
SAMPLE_SEED = 42  # Noneにすると、実行ごとに表示が変わる

preview = amounts.sample(n=5, random_state=SAMPLE_SEED).sample(
    n=12, axis=1, random_state=SAMPLE_SEED
)
with pd.option_context("display.max_columns", None):
    display(preview)

## 2. 前処理とWard法

In [ ]:
# 各都市の食費総額に対する割合に変換
food_total = pd.to_numeric(data["LB00"])
shares = amounts.div(food_total, axis=0)

# 品目ごとに標準化
X = StandardScaler().fit_transform(shares)

# 全47都市・212品目でWard法
Z = linkage(X, method="ward", metric="euclidean")

## 3. 樹形図を見る

In [ ]:
# 地方は色分けだけに使い、分類には使わない
region_cities = {
    "北海道・東北": "札幌市 青森市 盛岡市 仙台市 秋田市 山形市 福島市",
    "関東": "水戸市 宇都宮市 前橋市 さいたま市 千葉市 東京都区部 横浜市",
    "中部": "新潟市 富山市 金沢市 福井市 甲府市 長野市 岐阜市 静岡市 名古屋市",
    "近畿": "津市 大津市 京都市 大阪市 神戸市 奈良市 和歌山市",
    "中国": "鳥取市 松江市 岡山市 広島市 山口市",
    "四国": "徳島市 高松市 松山市 高知市",
    "九州": "福岡市 佐賀市 長崎市 熊本市 大分市 宮崎市 鹿児島市",
    "沖縄": "那覇市",
}
region_colors = dict(zip(region_cities, [
    "tab:blue", "tab:orange", "tab:green", "tab:red",
    "tab:purple", "tab:brown", "tab:pink", "tab:cyan",
]))
city_region = {
    city: region
    for region, cities in region_cities.items()
    for city in cities.split()
}
assert set(amounts.index) <= city_region.keys(), "地方が未設定の都市があります。"

# 同じ地方だけからなる枝は同色、複数の地方が混ざる枝は灰色
n = len(amounts)
node_region = [city_region[city] for city in amounts.index]
leaf_heights = {}
for left, right, height, _ in Z:
    left, right = int(left), int(right)
    a, b = node_region[left], node_region[right]
    node_region.append(a if a == b else None)
    for child in (left, right):
        if child < n:
            leaf_heights[child] = height

N_CLUSTERS = 6  # 分けるグループ数（2〜47）
assert isinstance(N_CLUSTERS, int) and 2 <= N_CLUSTERS <= n

# 樹形図を指定数に分け、上からG1、G2、...とする
original_groups = cut_tree(Z, n_clusters=[N_CLUSTERS]).ravel()
order = leaves_list(Z)
group_numbers = {
    group: i + 1
    for i, group in enumerate(dict.fromkeys(original_groups[order]))
}
groups = [group_numbers[group] for group in original_groups]

lower_height = 0.0 if N_CLUSTERS == n else Z[n - N_CLUSTERS - 1, 2]
upper_height = Z[n - N_CLUSTERS, 2]
cut_height = (lower_height + upper_height) / 2

fig, ax = plt.subplots(figsize=(11, 13))
tree = dendrogram(
    Z,
    labels=[f"{city}  [G{group}]" for city, group in zip(amounts.index, groups)],
    orientation="right",
    leaf_font_size=10,
    link_color_func=lambda k: region_colors.get(node_region[k], "0.65"),
    ax=ax,
)

# 都市名と末端の枝を地方の色にする
for tick, y, leaf in zip(ax.get_yticklabels(), ax.get_yticks(), tree["leaves"]):
    color = region_colors[city_region[amounts.index[leaf]]]
    tick.set_color(color)
    ax.plot([0, leaf_heights[leaf]], [y, y], color=color, linewidth=1.5)

# グループを分ける位置に線を引く
if upper_height > lower_height:
    ax.axvline(cut_height, color="0.25", linestyle="--", linewidth=1.2)
else:
    print("同じ高さの結合があるため、区切り線は引かず[G番号]で表示します。")

ax.invert_yaxis()
ax.set_title(f"食品への支出割合による都市の分類（Ward法・{N_CLUSTERS}グループ）")
ax.set_xlabel("結合距離（Ward距離）")
fig.tight_layout()
plt.show()